### *This file is for the quantification of cFos positive cells in striatal sections*

### *step 0: initialization*
In this step we load the functions we'll be using and assign some of them nicknames, as well as setting some constants if necessary. 

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
from datetime import datetime
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import stereo_helpers as hp
import pickle
# %matplotlib widget
%matplotlib inline

### *step 1: the loader*
In this step we'll load all the file types from the experiment for all subjects in the main directory.

In [ ]:
all_data = hp.load_pickle('../data/cfos/fos_cell_counts_flat.pkl')

all_data.index.is_monotonic_increasing

### *step 2 & 3: filterring and formating*

filter the df and add columns for normalized intensity by subject-section combination, y_inverted, and x_flipped

In [4]:
# Remove rows where subject_id is 'neg1' and section_number is '003'
all_data = all_data[~((all_data['subject_id'] == 'neg1') & (all_data['section_number'] == '003'))].reset_index(drop=True)

In [5]:
all_data.head()

,subject_id,exp_group,section_number,cell_number,fos_count,fos_int_intensity,fos_mean_intensity,fos_median_intensity,x_location,y_location
0,c577m11,coc_30mgKg,001,1,0,7.788237,0.027716,0.027451,985.309609,45.355872
1,c577m11,coc_30mgKg,001,2,0,5.717648,0.027357,0.027451,1008.344498,42.875598
2,c577m11,coc_30mgKg,001,3,0,5.984314,0.018760,0.015686,952.868339,53.147335
3,c577m11,coc_30mgKg,001,4,0,4.317648,0.024814,0.023529,932.419540,58.390805
4,c577m11,coc_30mgKg,001,5,0,3.694118,0.018471,0.015686,896.010000,56.240000


In [5]:
# Group by 'subject_id' and 'section_number' and then invert the y-coordinates within each group
all_data['y_inverted'] = all_data.groupby(['subject_id', 'section_number'])['y_location'].transform(lambda y: y.max() - y)

# Create a new column for flipped x-coordinates with the original x-coordinates
all_data['x_inverted'] = all_data['x_location']

# Define the groups that require flipping the x-coordinate
flip_groups = {
    'c577m11': ['002', '004'],
    'neg1': ['004'],
    'wt1': ['004'],
    'wt3': ['005']
}

# Calculate the max x-coordinate for each group and flip the x-coordinates for the specified groups
for subject, sections in flip_groups.items():
    for section in sections:
        # Calculate the max x-coordinate for the current subject and section
        max_x = all_data[(all_data['subject_id'] == subject) & (all_data['section_number'] == section)]['x_location'].max()
        # Select the rows that match the current subject and section and apply the flipping operation
        all_data.loc[(all_data['subject_id'] == subject) & (all_data['section_number'] == section), 'x_inverted'] = max_x - all_data['x_inverted']


# Calculate the median x and y coordinates for each group
medians = all_data.groupby(['subject_id', 'section_number']).agg({'x_inverted': 'median', 'y_inverted': 'median'})
medians = medians.rename(columns={'x_inverted': 'mid_x', 'y_inverted': 'mid_y'})

# Merge the median values back into the original dataframe
all_data = all_data.merge(medians, on=['subject_id', 'section_number'])

# Define the conditions for the regions using vectorized operations
conditions = [
    (all_data['x_inverted'] >= all_data['mid_x']) & (all_data['y_inverted'] <= all_data['mid_y']),
    (all_data['x_inverted'] >= all_data['mid_x']) & (all_data['y_inverted'] > all_data['mid_y']),
    (all_data['x_inverted'] < all_data['mid_x']) & (all_data['y_inverted'] > all_data['mid_y']),
    (all_data['x_inverted'] < all_data['mid_x']) & (all_data['y_inverted'] <= all_data['mid_y'])
]

# Define the region names corresponding to the conditions
regions = ['vls', 'dls', 'dms', 'vms']

# Assign regions based on the conditions using vectorized selection
all_data['region'] = np.select(conditions, regions)

# Optionally, drop the mid_x and mid_y columns if they are no longer needed
all_data.drop(columns=['mid_x', 'mid_y'], inplace=True)

### UNLESS RE-RUN CELLPROFILER ON SECTION '003' FOR neg1, REMOVE THE ROWS FOR THIS SECTION FOR THIS SUBJECT:
fos_data = all_data[all_data['region']!= '0'].reset_index(drop=True)

subject_order = ['wt1','neg1', 'wt2', 'c577m3','wt3', 'c577m11']


In [6]:

# Create a new dataframe 'fos_data' by copying the original dataframe
fos_data = all_data.copy()

#Keep only the cells where 'fos_count' is 1
# fos_data = fos_data[fos_data['fos_count']==1].reset_index(drop=True)

# # zero the intensity where the fos count is 0
fos_data.loc[fos_data['fos_count']==0,'fos_int_intensity'] = 0
fos_data.loc[fos_data['fos_count']>1,'fos_count'] = 0

# Calculate the max 'fos_int_intensity' for each group
max_intensity_by_group = fos_data.groupby('subject_id')['fos_int_intensity'].transform('max')

# Create a new column 'normalized_intensity' by dividing 'fos_int_intensity' by the max value
fos_data['normalized_intensity'] = fos_data['fos_int_intensity'] / max_intensity_by_group



### *step 4 & 5: computations and visualizations*

##### **4a & 5a: cfos visualization on striatal cells, and intensity spectrum**

5a: plot the all cells and cfos positive cells in scatter plots

##### **4b and 5b: fraction of cfos+ cells per region**

calculate fraction of cells that are cfos positive in each of the striatal region:

In [7]:
import pandas as pd

# Assuming 'df' is your DataFrame
# First, calculate the number of positive cells per region and section
positive_counts = fos_data[fos_data['fos_count'] == 1].groupby(['subject_id', 'exp_group', 'section_number', 'region']).size().reset_index(name='positive_cell_count')

# Then, calculate the total number of cells per section
total_counts = fos_data.groupby(['subject_id', 'exp_group', 'section_number']).size().reset_index(name='total_cell_count')

# Merge the two DataFrames on the 'section_number' column to align positive and total counts
merged_df = pd.merge(positive_counts, total_counts, on=['subject_id', 'exp_group', 'section_number'])

# Calculate the fraction for each region in a section
merged_df['fraction'] = merged_df['positive_cell_count'] / merged_df['total_cell_count']

# The resulting DataFrame contains the fraction of positive cells per region for each section
print(merged_df[['subject_id', 'exp_group', 'section_number', 'region', 'fraction']])

    subject_id   exp_group section_number region  fraction
0      c577m11  coc_30mgKg            001    dls  0.020874
1      c577m11  coc_30mgKg            001    dms  0.045049
2      c577m11  coc_30mgKg            001    vls  0.024563
3      c577m11  coc_30mgKg            001    vms  0.013883
4      c577m11  coc_30mgKg            002    dls  0.021001
..         ...         ...            ...    ...       ...
107        wt3  coc_30mgKg            005    vms  0.007201
108        wt3  coc_30mgKg            006    dls  0.017566
109        wt3  coc_30mgKg            006    dms  0.033974
110        wt3  coc_30mgKg            006    vls  0.048934
111        wt3  coc_30mgKg            006    vms  0.005694

[112 rows x 5 columns]


In [ ]:
sns.set_context('paper')

# Only keep VLS and DMS
region_order = ['vls', 'dms']
exp_group_order = ['control', 'coc_20mgKg', 'coc_30mgKg']

# Custom palette with requested colors
palette = {'control': '#808080', 'coc_20mgKg': '#ff8c00', 'coc_30mgKg': '#990000'}

# Convert mm to inches for matplotlib (1 inch = 25.4 mm)
width_in = 34.362 / 25.4
height_in = 65 / 25.4
plt.figure(figsize=(width_in, height_in))

# Filter merged_df for only VLS and DMS
plot_df = merged_df[merged_df['region'].isin(region_order)]

# Strip plot
sns.stripplot(
    data=plot_df, x='region', y='fraction', hue='exp_group',
    order=region_order, hue_order=exp_group_order, palette=palette,
    dodge=True, jitter=True, marker='o', alpha=0.5, size=5
)

# Point plot (mean)
sns.pointplot(
    data=plot_df, x='region', y='fraction', hue='exp_group',
    order=region_order, hue_order=exp_group_order,
    dodge=0.5, join=False, markers='_', ci=None, color='black', scale=2.0
)

# Remove legend
plt.legend([], [], frameon=False)

# Labels and formatting
plt.ylim(0, 0.1)
plt.xlabel('Region')
plt.ylabel('Mean Fraction')
# plt.title('Mean Fraction by Region and Experimental Group')
sns.despine()
plt.savefig('../output/cFos/mean_fraction_by_region_and_exp_group.pdf', dpi=300, bbox_inches='tight')
plt.show()

In [10]:
merged_df['subject_id'].unique()

array(['c577m11', 'c577m3', 'neg1', 'wt1', 'wt2', 'wt3'], dtype=object)

In [ ]:
# Statistical comparison of mean fractions between groups for each region (non-parametric tests)
import scipy.stats as stats
import statsmodels.stats.multitest as smm

region_order = ['vls', 'dms']
group_order = ['control', 'coc_20mgKg', 'coc_30mgKg']

results = []
output_lines = []
for region in region_order:
    region_df = merged_df[merged_df['region'] == region]
    # Print n number of sections for each group
    output_lines.append(f"\nRegion: {region}")
    print(f"\nRegion: {region}")
    for group in group_order:
        n_sections = region_df[region_df['exp_group'] == group].shape[0]
        output_lines.append(f"{group}: n={n_sections}")
        print(f"{group}: n={n_sections}")
    # Prepare data for Kruskal-Wallis test
    data_groups = [region_df[region_df['exp_group'] == group]['fraction'] for group in group_order]
    # Kruskal-Wallis test across all groups
    kw_stat, kw_p = stats.kruskal(*data_groups)
    output_lines.append(f"Kruskal-Wallis H={kw_stat:.3f}, p={kw_p:.4f}")
    print(f"Kruskal-Wallis H={kw_stat:.3f}, p={kw_p:.4f}")
    # Pairwise Mann-Whitney U tests
    pairs = [(group_order[0], group_order[1]), (group_order[0], group_order[2]), (group_order[1], group_order[2])]
    pvals = []
    for g1, g2 in pairs:
        x1 = region_df[region_df['exp_group'] == g1]['fraction']
        x2 = region_df[region_df['exp_group'] == g2]['fraction']
        u_stat, p = stats.mannwhitneyu(x1, x2, alternative='two-sided')
        pvals.append(p)
        output_lines.append(f"Mann-Whitney U test: {g1} vs {g2}: U={u_stat:.1f}, p={p:.4f}")
        print(f"Mann-Whitney U test: {g1} vs {g2}: U={u_stat:.1f}, p={p:.4f}")
    # Multiple comparison correction (FDR)
    adj_pvals = smm.multipletests(pvals, method='fdr_bh')[1]
    for i, (g1, g2) in enumerate(pairs):
        output_lines.append(f"Adjusted p-value for {g1} vs {g2}: {adj_pvals[i]:.4f}")
        print(f"Adjusted p-value for {g1} vs {g2}: {adj_pvals[i]:.4f}")
    results.append({'region': region, 'kruskal_p': kw_p, 'pairwise_p': adj_pvals})

# Option to save results to txt file
save_results = True  # Set to True to save output
if save_results:
    with open('../output/cFos/nonparametric_stats_results_cFos_counts.txt', 'w') as f:
        for line in output_lines:
            f.write(line + '\n')

# Results are printed for each region; you can use these p-values to annotate your plots.

In [ ]:
# Parametric statistical comparison of mean fractions between groups for each region (ANOVA and t-tests)
import scipy.stats as stats
from statsmodels.stats.multitest import multipletests

region_order = ['vls', 'dms']
group_order = ['control', 'coc_20mgKg', 'coc_30mgKg']

results_parametric = []
output_lines = []
for region in region_order:
    region_df = merged_df[merged_df['region'] == region]
    # Print n number of sections for each group
    output_lines.append(f"\nRegion: {region}")
    print(f"\nRegion: {region}")
    for group in group_order:
        n_sections = region_df[region_df['exp_group'] == group].shape[0]
        output_lines.append(f"{group}: n={n_sections}")
        print(f"{group}: n={n_sections}")
    # Prepare data for ANOVA
    data_groups = [region_df[region_df['exp_group'] == group]['fraction'] for group in group_order]
    # One-way ANOVA across all groups
    f_stat, f_p = stats.f_oneway(*data_groups)
    output_lines.append(f"One-way ANOVA F={f_stat:.3f}, p={f_p:.4f}")
    print(f"One-way ANOVA F={f_stat:.3f}, p={f_p:.4f}")
    # Pairwise t-tests
    pairs = [(group_order[0], group_order[1]), (group_order[0], group_order[2]), (group_order[1], group_order[2])]
    pvals = []
    for g1, g2 in pairs:
        x1 = region_df[region_df['exp_group'] == g1]['fraction']
        x2 = region_df[region_df['exp_group'] == g2]['fraction']
        t_stat, p = stats.ttest_ind(x1, x2, equal_var=False)
        pvals.append(p)
        output_lines.append(f"t-test: {g1} vs {g2}: t={t_stat:.2f}, p={p:.4f}")
        print(f"t-test: {g1} vs {g2}: t={t_stat:.2f}, p={p:.4f}")
    # Multiple comparison correction (FDR)
    adj_pvals = multipletests(pvals, method='fdr_bh')[1]
    for i, (g1, g2) in enumerate(pairs):
        output_lines.append(f"Adjusted p-value for {g1} vs {g2}: {adj_pvals[i]:.4f}")
        print(f"Adjusted p-value for {g1} vs {g2}: {adj_pvals[i]:.4f}")
    results_parametric.append({'region': region, 'anova_p': f_p, 'pairwise_p': adj_pvals})

# Option to save results to txt file
save_results = True  # Set to True to save output
if save_results:
    with open('../output/cFos/parametric_stats_results_cFos_counts.txt', 'w') as f:
        for line in output_lines:
            f.write(line + '\n')

# Results are printed for each region; you can use these p-values to annotate your plots.

In [ ]:
# Plot fold difference increase of each dose compared to control for each region
import matplotlib.pyplot as plt
import numpy as np

region_order = ['vls', 'dms']
group_order = ['control', 'coc_20mgKg', 'coc_30mgKg']

fold_changes = {region: {} for region in region_order}
pvals = {region: {} for region in region_order}

for region in region_order:
    region_df = merged_df[merged_df['region'] == region]
    means = region_df.groupby('exp_group')['fraction'].mean()
    # Calculate fold change vs control
    for dose in group_order[1:]:
        fold = means[dose] / means['control'] if means['control'] != 0 else np.nan
        fold_changes[region][dose] = fold
        # t-test for parametric, Mann-Whitney for non-parametric
        x_control = region_df[region_df['exp_group'] == 'control']['fraction']
        x_dose = region_df[region_df['exp_group'] == dose]['fraction']
        # Use Mann-Whitney U test (non-parametric)
        stat, p = stats.mannwhitneyu(x_control, x_dose, alternative='two-sided')
        pvals[region][dose] = p

# Plotting
fig, ax = plt.subplots(figsize=(6, 4))
bar_width = 0.35
x = np.arange(len(region_order))
colors = ['#ff8c00', '#990000']
labels = ['coc_20mgKg', 'coc_30mgKg']

for i, dose in enumerate(labels):
    values = [fold_changes[region][dose] for region in region_order]
    ax.bar(x + i*bar_width, values, bar_width, label=f'{dose} vs control', color=colors[i], alpha=0.7)
    # Annotate p-values
    for j, region in enumerate(region_order):
        p = pvals[region][dose]
        ax.text(x[j] + i*bar_width, values[j], f'p={p:.3f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x + bar_width/2)
ax.set_xticklabels(region_order)
ax.set_ylabel('Fold Change vs Control')
ax.set_title('Fold Increase of Fraction (Dose vs Control)')
ax.legend()
sns.despine()
plt.tight_layout()
plt.savefig('../output/cFos/fold_change_vs_control_by_region.pdf', dpi=300)
plt.show()

# Print fold changes and p-values
for region in region_order:
    for dose in labels:
        print(f"{region}: {dose} fold change = {fold_changes[region][dose]:.2f}, p={pvals[region][dose]:.4f}")

In [ ]:
# Compare fold change distributions between VLS and DMS for each dose using Mann-Whitney U test
import numpy as np
import scipy.stats as stats

group_order = ['control', 'coc_20mgKg', 'coc_30mgKg']
region_order = ['vls', 'dms']

results = []
for dose in group_order[1:]:
    # Calculate mean control fraction for each region
    mean_control = {}
    for region in region_order:
        control_df = merged_df[(merged_df['region'] == region) & (merged_df['exp_group'] == 'control')]
        mean_control[region] = control_df['fraction'].mean()
    # Calculate fold change for each section in VLS and DMS for this dose
    vls_fold = merged_df[(merged_df['region'] == 'vls') & (merged_df['exp_group'] == dose)].copy()
    dms_fold = merged_df[(merged_df['region'] == 'dms') & (merged_df['exp_group'] == dose)].copy()
    vls_fold['fold_change'] = vls_fold['fraction'] / mean_control['vls'] if mean_control['vls'] != 0 else np.nan
    dms_fold['fold_change'] = dms_fold['fraction'] / mean_control['dms'] if mean_control['dms'] != 0 else np.nan
    # Mann-Whitney U test between VLS and DMS fold changes
    stat, p = stats.mannwhitneyu(vls_fold['fold_change'], dms_fold['fold_change'], alternative='two-sided')
    print(f"\nDose: {dose}")
    print(f"VLS fold changes: {vls_fold['fold_change'].values}")
    print(f"DMS fold changes: {dms_fold['fold_change'].values}")
    print(f"Mann-Whitney U test between VLS and DMS fold changes: U={stat:.2f}, p={p:.4f}")
    results.append({'dose': dose, 'U': stat, 'p': p})

# Optionally, save results to txt file
save_results = True
if save_results:
    with open('../output/cFos/fold_change_VLS_vs_DMS_stats.txt', 'w') as f:
        for r in results:
            f.write(f"Dose: {r['dose']}, Mann-Whitney U={r['U']:.2f}, p={r['p']:.4f}\n")

In [ ]:
# Plot mean fold change per region/dose with SEM error bars (per section), with Illustrator-compatible editable text boxes
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.font_manager import FontProperties

# Set rcParams for Illustrator compatibility
mpl.rcParams['pdf.use14corefonts'] = True
mpl.rcParams['pdf.fonttype'] = 42

region_order = ['vls', 'dms']
dose_order = ['coc_20mgKg', 'coc_30mgKg']

# Calculate mean control fraction for each region
mean_control = {}
for region in region_order:
    control_df = merged_df[(merged_df['region'] == region) & (merged_df['exp_group'] == 'control')]
    mean_control[region] = control_df['fraction'].mean()

# For each region and dose, calculate fold change per section
fold_change_data = {region: {dose: [] for dose in dose_order} for region in region_order}
for region in region_order:
    for dose in dose_order:
        dose_df = merged_df[(merged_df['region'] == region) & (merged_df['exp_group'] == dose)]
        # Calculate fold change for each section
        fold_changes = dose_df['fraction'] / mean_control[region] if mean_control[region] != 0 else np.nan
        fold_change_data[region][dose] = fold_changes.values

# Prepare data for plotting
means = np.zeros((len(region_order), len(dose_order)))
sems = np.zeros((len(region_order), len(dose_order)))
for i, region in enumerate(region_order):
    for j, dose in enumerate(dose_order):
        vals = fold_change_data[region][dose]
        means[i, j] = np.nanmean(vals)
        sems[i, j] = np.nanstd(vals, ddof=1) / np.sqrt(np.sum(~np.isnan(vals)))

# Font properties for all text
arial_fp = FontProperties(family='Arial', size=8)

# Plotting
fig, ax = plt.subplots(figsize=(2, 3))
bar_width = 0.35
x = np.arange(len(region_order))
colors = ['#ff8c00', '#990000']

for j, dose in enumerate(dose_order):
    # Plot bars with outlines only
    bars = ax.bar(x + j*bar_width, means[:, j], bar_width, yerr=sems[:, j], label=f'{dose} vs control', facecolor='none', edgecolor=colors[j], alpha=0.7, capsize=5)
    
    # Plot individual data points as empty circles
    for i, region in enumerate(region_order):
        vals = fold_change_data[region][dose]
        # Add jitter to x-axis for better visibility
        jitter = np.random.normal(0, 0.04, size=len(vals))
        ax.scatter(x[i] + j*bar_width + jitter, vals, facecolors='none', edgecolors=colors[j], marker='o', alpha=0.6)

    # Add mean value as a text box above each bar (editable in Illustrator)
    for i, bar in enumerate(bars):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, height + sems[i,j], f'{height:.2f}', ha='center', va='bottom', fontproperties=arial_fp, bbox=dict(facecolor='white', edgecolor='none', boxstyle='round,pad=0.2'))

ax.set_xticks(x + bar_width/2)
ax.set_xticklabels(region_order, fontproperties=arial_fp)
ax.set_ylabel('Fold Change vs Control', fontproperties=arial_fp)
ax.set_xlabel('Region', fontproperties=arial_fp)
ax.set_title('Mean Fold Increase of Fraction (Dose vs Control) with SEM', fontproperties=FontProperties(family='Arial', size=10))
ax.legend(fontsize=8, frameon=False, prop=arial_fp)
ax.tick_params(axis='x', labelsize=8)
ax.tick_params(axis='y', labelsize=8)
sns.despine()
plt.tight_layout()
plt.savefig('../output/cFos/fold_change_vs_control_by_region_with_SEM.pdf', dpi=300)
plt.show()

# Print means and SEMs
for i, region in enumerate(region_order):
    for j, dose in enumerate(dose_order):
        print(f"{region}: {dose} mean fold change = {means[i, j]:.2f}, SEM = {sems[i, j]:.2f}")

In [ ]:
# Compare fold change distributions between VLS and DMS for each dose using parametric t-tests
import numpy as np
import scipy.stats as stats

group_order = ['control', 'coc_20mgKg', 'coc_30mgKg']
region_order = ['vls', 'dms']

results = []
for dose in group_order[1:]:
    # Calculate mean control fraction for each region
    mean_control = {}
    for region in region_order:
        control_df = merged_df[(merged_df['region'] == region) & (merged_df['exp_group'] == 'control')]
        mean_control[region] = control_df['fraction'].mean()
    # Calculate fold change for each section in VLS and DMS for this dose
    vls_fold = merged_df[(merged_df['region'] == 'vls') & (merged_df['exp_group'] == dose)].copy()
    dms_fold = merged_df[(merged_df['region'] == 'dms') & (merged_df['exp_group'] == dose)].copy()
    vls_fold['fold_change'] = vls_fold['fraction'] / mean_control['vls'] if mean_control['vls'] != 0 else np.nan
    dms_fold['fold_change'] = dms_fold['fraction'] / mean_control['dms'] if mean_control['dms'] != 0 else np.nan
    # Parametric t-test between VLS and DMS fold changes
    t_stat, p = stats.ttest_ind(vls_fold['fold_change'], dms_fold['fold_change'], equal_var=False)
    print(f"\nDose: {dose}")
    print(f"VLS fold changes: {vls_fold['fold_change'].values}")
    print(f"DMS fold changes: {dms_fold['fold_change'].values}")
    print(f"t-test between VLS and DMS fold changes: t={t_stat:.2f}, p={p:.4f}")
    results.append({'dose': dose, 't': t_stat, 'p': p})

# Optionally, save results to txt file
save_results = True
if save_results:
    with open('../output/cFos/fold_change_VLS_vs_DMS_parametric_stats.txt', 'w') as f:
        for r in results:
            f.write(f"Dose: {r['dose']}, t-test t={r['t']:.2f}, p={r['p']:.4f}\n")

## Stats table — updated format

| outcome | Design/Test | Unit (n) | Statistic (df) | Effect size (Cohen's d) | 95% CI for d | P (adjusted, Bonferroni ×2) |
|---|---|---:|---|---:|---|---:|
| VLS fold change vs DMS fold change (Dose: 20 mg/kg) | Two‑sample t‑test (two‑tailed, Student's t), equal variances assumed | Control n = 9; Coc 20 mg/kg n = 9 | t(16) = −2.160 | −1.018 | (−1.992, −0.016) | p_raw = 0.04629 → p_adj = 0.09258 |
| VLS fold change vs DMS fold change (Dose: 30 mg/kg) | Two‑sample t‑test (two‑tailed, Student's t), equal variances assumed | Control n = 9; Coc 30 mg/kg n = 10 | t(17) = 3.280 | 1.507 | (0.459, 2.522) | p_raw = 0.00442 → p_adj = 0.00883 |

**Notes:**

- P values are two‑tailed. Bonferroni adjustment applied for 2 planned pairwise comparisons (×2). If you used a different family size or correction (Holm, FDR), tell me and I'll update the adjusted p-values.
- Cohen's d computed from reported t using d = t * sqrt(1/n1 + 1/n2).
- 95% CIs for d were obtained by converting the noncentral‑t CI for the noncentrality parameter to d (solver via SciPy).
- If you prefer Hedges' g (small‑sample correction) or a different CI method, I can compute that as well.

In [2]:
# Recreate the stats used for the table: p-values, Cohen's d, and 95% CIs for d
import math
from scipy import stats, optimize
import pandas as pd

def compute_stats(n1, n2, t_obs, comparisons=1, alpha=0.05):
    df = n1 + n2 - 2
    p_raw = stats.t.sf(abs(t_obs), df) * 2
    p_adj = min(1.0, p_raw * comparisons)
    # Cohen's d (pooled) derived from t
    d = t_obs * math.sqrt(1.0/n1 + 1.0/n2)

    # Noncentral-t inversion to get CI for noncentrality parameter (delta)
    def delta_ci(t_obs, df, alpha=0.05):
        def f_low(delta):
            return stats.nct.cdf(t_obs, df, delta) - alpha/2
        def f_high(delta):
            return stats.nct.cdf(t_obs, df, delta) - (1 - alpha/2)
        low_bound, high_bound = -100.0, 100.0
        try:
            delta_low = optimize.brentq(f_low, low_bound, high_bound, maxiter=200)
        except Exception:
            delta_low = float('nan')
        try:
            delta_high = optimize.brentq(f_high, low_bound, high_bound, maxiter=200)
        except Exception:
            delta_high = float('nan')
        return delta_low, delta_high

    delta_low, delta_high = delta_ci(t_obs, df, alpha=alpha)
    factor = math.sqrt(1.0/n1 + 1.0/n2)
    d_ci = (delta_low * factor, delta_high * factor)

    return {
        'n1': n1, 'n2': n2, 'df': df, 't': t_obs,
        'p_raw': p_raw, 'p_adj': p_adj,
        'd': d, 'd_ci': d_ci
    }

# Example inputs (from the manuscript table)
cases = [
    {'label': 'Dose: 20 mg/kg', 'n1': 9, 'n2': 9, 't': -2.16},
    {'label': 'Dose: 30 mg/kg', 'n1': 9, 'n2': 10, 't': 3.28},
]

rows = []
for c in cases:
    r = compute_stats(c['n1'], c['n2'], c['t'], comparisons=2, alpha=0.05)
    rows.append({
        'outcome': f"VLS vs DMS ({c['label']})",
        'design_test': "Two-sample t-test (two-tailed, Student's t)",
        'unit_n': f"Control n={c['n1']}; Coc n={c['n2']}",
        'statistic': f"t({r['df']}) = {r['t']:.3f}",
        'cohens_d': round(r['d'], 3),
        'd_95_CI': (round(r['d_ci'][0], 3), round(r['d_ci'][1], 3)),
        'p_raw': round(r['p_raw'], 6),
        'p_adj': round(r['p_adj'], 6)
    })

df = pd.DataFrame(rows)

# Fallback markdown printer to avoid external 'tabulate' dependency
def df_to_markdown(df):
    cols = list(df.columns)
    header = '| ' + ' | '.join(cols) + ' |'
    sep = '| ' + ' | '.join(['---']*len(cols)) + ' |'
    rows_md = []
    for _, r in df.iterrows():
        vals = []
        for v in r.tolist():
            if isinstance(v, tuple):
                vals.append(f"({v[0]}, {v[1]})")
            else:
                vals.append(str(v))
        rows_md.append('| ' + ' | '.join(vals) + ' |')
    return '\n'.join([header, sep] + rows_md)

print(df_to_markdown(df))
# Optionally save to CSV:
# df.to_csv('stats_table_recreated.csv', index=False)


| outcome | design_test | unit_n | statistic | cohens_d | d_95_CI | p_raw | p_adj |
| --- | --- | --- | --- | --- | --- | --- | --- |
| VLS vs DMS (Dose: 20 mg/kg) | Two-sample t-test (two-tailed, Student's t) | Control n=9; Coc n=9 | t(16) = -2.160 | -1.018 | (-0.016, -1.992) | 0.046292 | 0.092584 |
| VLS vs DMS (Dose: 30 mg/kg) | Two-sample t-test (two-tailed, Student's t) | Control n=9; Coc n=10 | t(17) = 3.280 | 1.507 | (2.522, 0.459) | 0.004417 | 0.008833 |


In [9]:
# Compute effect sizes and CIs for the parametric pairwise t-tests and print final table
import math
import numpy as np
import scipy.stats as stats
from statsmodels.stats.multitest import multipletests

regions = ['vls', 'dms']
groups = ['control', 'coc_20mgKg', 'coc_30mgKg']

rows = []
for region in regions:
    region_df = merged_df[merged_df['region'] == region]
    # collect data arrays per group
    data = {g: region_df[region_df['exp_group'] == g]['fraction'].values for g in groups}
    # ANOVA
    f_stat, f_p = stats.f_oneway(*[data[g] for g in groups])
    # pairwise
    pairs = [(groups[0], groups[1]), (groups[0], groups[2]), (groups[1], groups[2])]
    pvals = []
    details = []
    for g1, g2 in pairs:
        x1 = data[g1]; x2 = data[g2]
        n1 = len(x1); n2 = len(x2)
        m1 = np.nanmean(x1); m2 = np.nanmean(x2)
        s1 = np.nanstd(x1, ddof=1); s2 = np.nanstd(x2, ddof=1)
        # Welch t-test
        t_stat, p = stats.ttest_ind(x1, x2, equal_var=False, nan_policy='omit')
        # Welch-Satterthwaite df
        se_sq1 = s1*s1 / n1
        se_sq2 = s2*s2 / n2
        denom = (se_sq1 + se_sq2)
        if denom == 0:
            welch_df = float('nan')
        else:
            welch_df = (se_sq1 + se_sq2)**2 / ((se_sq1**2)/(n1-1) + (se_sq2**2)/(n2-1))
        # Cohen's d (pooled)
        pooled_sd = math.sqrt(((n1-1)*s1*s1 + (n2-1)*s2*s2) / (n1 + n2 - 2)) if (n1+n2-2)>0 else float('nan')
        d = (m1 - m2) / pooled_sd if pooled_sd>0 else float('nan')
        # Hedges' g correction
        df_total = n1 + n2 - 2
        J = 1 - (3.0/(4.0*df_total - 1.0)) if df_total>1 else 1.0
        g = d * J
        # approximate var for d (Hedges & Olkin)
        var_d = (n1 + n2)/(n1 * n2) + d**2/(2.0 * (n1 + n2))
        se_d = math.sqrt(var_d)
        ci_low = d - 1.96 * se_d
        ci_high = d + 1.96 * se_d
        pvals.append(p)
        details.append({'g1': g1, 'g2': g2, 'n1': n1, 'n2': n2, 't': t_stat, 'df': welch_df, 'd': d, 'g': g, 'ci': (ci_low, ci_high), 'p': p})
    # FDR adjust per-region pairwise p-values (same as notebook)
    adj = multipletests(pvals, method='fdr_bh')[1]
    for i, det in enumerate(details):
        det['p_adj'] = adj[i]
        rows.append({
            'outcome': f"{region.upper()}: {det['g1']} vs {det['g2']}",
            'design_test': "Welch t-test (two-tailed)",
            'unit_n': f"{det['g1']} n={det['n1']}; {det['g2']} n={det['n2']}",
            'statistic': f"t({det['df']:.2f}) = {det['t']:.3f}",
            'effect_size_d': round(det['d'], 3),
            '95_CI_d': (round(det['ci'][0], 3), round(det['ci'][1], 3)),
            'p_adj': round(det['p_adj'], 6)
        })
    # also include ANOVA row
    eta2 = f_stat * (len(region_df) - len(groups)) / ( (len(region_df) - len(groups)) + (len(groups)-1) * f_stat ) if (len(region_df)>0) else float('nan')
    rows.append({
        'outcome': f"{region.upper()}: One-way ANOVA",
        'design_test': 'One-way ANOVA',
        'unit_n': ', '.join([f"{g} n={len(data[g])}" for g in groups]),
        'statistic': f"F = {f_stat:.3f}, p = {f_p:.4f}",
        'effect_size_d': round(eta2, 3),
        '95_CI_d': ('-', '-'),
        'p_adj': round(f_p, 6)
    })

# print markdown table
cols = ['outcome','design_test','unit_n','statistic','effect_size_d','95_CI_d','p_adj']
print('| ' + ' | '.join(cols) + ' |')
print('| ' + ' | '.join(['---']*len(cols)) + ' |')
for r in rows:
    ci = r['95_CI_d']
    print(f"| {r['outcome']} | {r['design_test']} | {r['unit_n']} | {r['statistic']} | {r['effect_size_d']} | ({ci[0]}, {ci[1]}) | {r['p_adj']} |")


| outcome | design_test | unit_n | statistic | effect_size_d | 95_CI_d | p_adj |
| --- | --- | --- | --- | --- | --- | --- |
| VLS: control vs coc_20mgKg | Welch t-test (two-tailed) | control n=9; coc_20mgKg n=9 | t(8.35) = -4.361 | -2.056 | (-3.198, -0.914) | 0.002173 |
| VLS: control vs coc_30mgKg | Welch t-test (two-tailed) | control n=9; coc_30mgKg n=10 | t(9.02) = -6.352 | -2.761 | (-4.018, -1.503) | 0.000395 |
| VLS: coc_20mgKg vs coc_30mgKg | Welch t-test (two-tailed) | coc_20mgKg n=9; coc_30mgKg n=10 | t(9.71) = -5.373 | -2.344 | (-3.513, -1.175) | 0.00052 |
| VLS: One-way ANOVA | One-way ANOVA | control n=9, coc_20mgKg n=9, coc_30mgKg n=10 | F = 30.743, p = 0.0000 | 8.887 | (-, -) | 0.0 |
| DMS: control vs coc_20mgKg | Welch t-test (two-tailed) | control n=9; coc_20mgKg n=9 | t(8.21) = -4.665 | -2.199 | (-3.369, -1.029) | 0.00226 |
| DMS: control vs coc_30mgKg | Welch t-test (two-tailed) | control n=9; coc_30mgKg n=10 | t(9.59) = -12.110 | -5.28 | (-7.185, -3.375) | 1e-06 |
| 